# *Model de clasificare a imaginilor OCT folosind ResNet18*
***Este implementat în PyTorch***

**1. Descărcare dataset**


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("paultimothymooney/kermany2018")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'kermany2018' dataset.
Path to dataset files: /kaggle/input/kermany2018


**2. Mutare dataset**

In [ ]:
import os
import shutil

BASE = "/content/oct"
RAW = os.path.join(BASE, "raw")

os.makedirs(BASE, exist_ok=True)

# Copiez datele
shutil.copytree(path, RAW, dirs_exist_ok=True)

print("RAW structure:")
for root, dirs, files in os.walk(RAW):
    print(root, dirs[:5])
    break


RAW structure:
/content/oct/raw ['OCT2017 ', 'oct2017']


`Iportări necesare`

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import time

**3. Preprocesare și pregătire date**

*a)Preprocesare*
* Redimensionare imagine la (224, 224)
* Flip aleator pe orizontală
* Transformare imagine în tensor PyTorch
* Normalizare pixelii după media și deviația standard ImageNet

*b)Pregătire date*

In [ ]:

IMAGE_SIZE = 224
BATCH_SIZE = 32

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),                            #resize
        transforms.RandomHorizontalFlip(),                                      #flip
        transforms.ToTensor(),                                                  #transformare în tensor
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])      #normalizare
    ]),
    'val': transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# seturi de date
image_datasets = {x: datasets.ImageFolder(
    os.path.join(DATASET_ROOT, x),
    data_transforms[x]
) for x in ['train', 'val']}

dataloaders = {x: DataLoader(
    image_datasets[x],
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
) for x in ['train', 'val']}

dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes
print(f"\nClasele: {class_names}")
print(f"Dimensiunea setului de antrenare: {dataset_sizes['train']}")
print(f"Dimensiunea setului de validare: {dataset_sizes['val']}")


Clasele: ['CNV', 'DME', 'DRUSEN', 'NORMAL']
Dimensiunea setului de antrenare: 83484
Dimensiunea setului de validare: 32


**4. Creare și compilare model**

In [ ]:
import torchvision.models as models

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"\nUtilizez dispozitivul: {device}")

# ResNet-18 pre-antrenat pe ImageNet
model_ft = models.resnet18(weights='IMAGENET1K_V1')


num_ftrs = model_ft.fc.in_features
model_ft.fc = nn.Linear(num_ftrs, len(class_names))

model_ft = model_ft.to(device)

criterion = nn.CrossEntropyLoss()
optimizer_ft = optim.Adam(model_ft.parameters(), lr=1e-4)


Utilizăm dispozitivul: cuda:0


`Verificare foldere`

In [ ]:
import os
from torchvision import datasets

DATASET_ROOT = "/content/oct/raw/OCT2017 /"

# Verific'train' și 'val'
size_train = len(datasets.ImageFolder(os.path.join(DATASET_ROOT, 'train')))
size_val = len(datasets.ImageFolder(os.path.join(DATASET_ROOT, 'val')))

print(f"Dimensiunea setului de antrenare: {size_train}")
print(f"Dimensiunea setului de validare: {size_val}")



Dimensiunea setului de antrenare: 83484
Dimensiunea setului de validare: 32


**5. Antrenare model**

In [ ]:


print("\n>>> Antrenare<<<")
final_model = train_model(
    model_ft,
    criterion,
    optimizer_ft,
    dataloaders,
    dataset_sizes,
    device,
    num_epochs=10
)
print("Antrenare finalizată.")


>>> Antrenarea va începe acum <<<
Epoch 0/9
----------
train Loss: 0.1924 Acc: 0.9333
val Loss: 0.0366 Acc: 1.0000

Epoch 1/9
----------
train Loss: 0.1128 Acc: 0.9610
val Loss: 0.0221 Acc: 1.0000

Epoch 2/9
----------
train Loss: 0.0914 Acc: 0.9682
val Loss: 0.0168 Acc: 1.0000

Epoch 3/9
----------
train Loss: 0.0753 Acc: 0.9730
val Loss: 0.0218 Acc: 1.0000

Epoch 4/9
----------
train Loss: 0.0642 Acc: 0.9773
val Loss: 0.0284 Acc: 0.9688

Epoch 5/9
----------
train Loss: 0.0544 Acc: 0.9808
val Loss: 0.0383 Acc: 0.9688

Epoch 6/9
----------
train Loss: 0.0472 Acc: 0.9832
val Loss: 0.0105 Acc: 1.0000

Epoch 7/9
----------
train Loss: 0.0415 Acc: 0.9855
val Loss: 0.0125 Acc: 1.0000

Epoch 8/9
----------
train Loss: 0.0354 Acc: 0.9874
val Loss: 0.0267 Acc: 0.9688

Epoch 9/9
----------
train Loss: 0.0316 Acc: 0.9888
val Loss: 0.0461 Acc: 0.9688

Antrenarea completă în 81m 20s
Cea mai bună acuratețe de validare: 1.0000
Antrenarea finalizată.


**3.** *c) Pregătire date testare*

In [ ]:
# Test dataset
data_transforms['test'] = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

image_datasets['test'] = datasets.ImageFolder(
    os.path.join(DATASET_ROOT, 'test'),
    data_transforms['test']
)

dataloaders['test'] = DataLoader(
    image_datasets['test'],
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

dataset_sizes['test'] = len(image_datasets['test'])
print("Dimensiunea test:", dataset_sizes['test'])


Dimensiunea test: 968


**6. Testare model**

In [ ]:
test_acc = evaluate_test_set(
    final_model,
    dataloaders['test'],
    dataset_sizes['test'],
    device,
    criterion
)


-------------------------
Începe Evaluarea Setului de Test...
Test Loss: 0.0268
Test Acuratețe: 0.9907
-------------------------


In [ ]:
torch.save(final_model, "oct_cnn_model_ResNet18.pth")
